<a href="https://colab.research.google.com/github/nuna-aa/extended-research/blob/main/multilingual_induction_heads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilingual Induction Heads

## What is this notebook about?

This notebook investigates **induction heads** -- a type of attention head inside a transformer language model that helps the model copy or continue repeated patterns in text.

For example, given the sequence `A B C A B`, an induction head helps the model predict `C` after the second `A B`, because it 'remembers' that `B` was followed by `C` the first time.

We study whether induction heads behave the **same way across different languages** (English and Yoruba), or whether different heads are responsible for the same mechanism in each language. This is part of a broader field called **mechanistic interpretability** -- the science of understanding what individual components of a neural network actually do.

### Key concepts for beginners

| Term | Plain-English meaning |
|---|---|
| **Attention head** | One of many parallel 'focus units' inside a transformer layer. Each head decides which earlier tokens to attend to when processing the current token. |
| **Induction head** | Looks back to find the last occurrence of the current token and attends to what followed it -- enabling in-context learning. |
| **Ablation** | Zeroing out one head's output to measure how much performance drops without it. Big drop = important head. |
| **Log-probability / Loss** | How confident the model is about the correct next token. Higher loss = worse performance. |
| **Mechanistic interpretability** | Reverse-engineering a neural network to understand the algorithm it has learned. |

### Notebook structure

1. **Setup** -- install packages and imports
2. **Configuration** -- choose model, languages, and dataset
3. **Data pipeline** -- load text and build repeated-token test sequences
4. **Induction score** -- score every attention head for induction-like behaviour
5. **Visualisation** -- inspect attention patterns of high-scoring heads
6. **Ablation study** -- measure how much each head contributes to model performance
7. **Cross-language comparison** -- compare which heads matter for English vs. Yoruba
8. **Cross-language ablation** -- test whether 'English heads' also matter for Yoruba and vice versa


---
## 1. Setup

Install all required packages. Only needs to run once per environment.

- `huggingface_hub` / `datasets` / `transformers` -- load models and datasets from HuggingFace
- `nnterp` -- uniform interface to many transformer architectures
- `torch` / `einops` -- tensor operations and easy reshaping
- `circuitsvis` -- interactive attention-pattern visualisations
- `plotly` -- heatmap charts


In [1]:
%pip install huggingface_hub torch nnterp[display] einops pandas circuitsvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 11.3 MB/s eta 0:00:00


Import everything we need and set a fixed random seed for reproducibility.

In [2]:
import huggingface_hub
from datasets import load_dataset
import pandas as pd
from transformers import AutoTokenizer
from nnterp import StandardizedTransformer
import torch
import einops
from typing import Tuple
import torch.nn.functional as F
import circuitsvis as cv
from IPython.display import clear_output
from tqdm import tqdm
import plotly.express as px
from collections import defaultdict
import gc

# Fix the random seed so results are reproducible across runs
torch.manual_seed(3407)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


---
## 2. Configuration

All parameters you might want to change are collected here in one place.

- **MODEL_IDS** -- maps a short nickname to the HuggingFace model path. We use `tiny-aya`, a small multilingual model by Cohere Labs, so that the notebook runs on modest hardware.
- **LANGUAGES** -- maps a short code to the language code used in the FLORES+ dataset. `eng_Latn` = English (Latin script), `yor_Latn` = Yoruba (Latin script).
- **DATASET** -- FLORES+ is a high-quality parallel translation benchmark covering 200+ languages.
- **SPLIT** -- we use `devtest`, the standard evaluation split.


In [3]:
# Model nicknames -> HuggingFace model paths
MODEL_IDS = {
    "tiny-aya": "CohereLabs/tiny-aya-global",
}

# Language short-codes -> FLORES+ language codes
LANGUAGES = {
    "eng": "eng_Latn",   # English
    "yor": "yor_Latn",   # Yoruba
}

# Dataset settings
DATASET = "openlanguagedata/flores_plus"
SPLIT   = "devtest"


---
## 3. Data Pipeline

To test induction heads we need carefully constructed input sequences. The pipeline has four stages:

```
FLORES+ text
    |
    v
load_hf_dataset()          <- download raw sentences per language
    |
    v
build_token_pools()        <- collect token IDs that appear in each language
    |
    v
build_repeated_sequence()  <- create sequences like [A B C A B C A B C ...]
    |
    v
prepend_bos_prefix()       <- add the model's BOS token at position 0
```

### Why repeated sequences?

Induction heads are identified by their tendency to attend back exactly `(seq_len - 1)` positions to find the previous occurrence of the current token. By repeating the same random token block `n_repeats` times, we create many instances of that pattern in every input, making the induction signal strong and easy to measure.


In [4]:
def load_hf_dataset(dataset_name: str, languages: dict, split: str) -> dict:
    """Download the FLORES+ dataset for each language and return a dict of DataFrames."""
    data = {}
    for lang_key, lang_code in languages.items():
        data[lang_key] = load_dataset(dataset_name, lang_code, split=split).to_pandas()
    return data


In [5]:
def load_models(model_ids: dict) -> dict:
    """
    Load each model from HuggingFace and wrap it with StandardizedTransformer,
    which gives a uniform API regardless of architecture differences.
    """
    models = {}
    for model_key, model_id in model_ids.items():
        model = StandardizedTransformer(model_id, device_map='auto', dispatch=True)
        clear_output()
        print(
            f'Loaded: {model_key}\n'
            f'  Device:          {model.device}\n'
            f'  Layers:          {model.num_layers}\n'
            f'  Hidden size:     {model.hidden_size}\n'
            f'  Attention heads: {model.num_heads}\n'
            f'  Vocabulary size: {model.vocab_size}'
        )
        models[model_key] = model
    return models


In [6]:
def build_token_pools(models: dict, language_data_dict: dict,
                      min_token: int = 1000, max_token: int = None) -> dict:
    """
    For each (model, language) pair, collect the set of "ordinary" token IDs
    that appear in that language's text.

    We filter out:
    - Very low IDs (< min_token): often byte-level or special tokens
    - Very high IDs (> vocab_size - 1000): often unused/reserved slots
    - Any special tokens (BOS, EOS, PAD, etc.) and the unknown token

    This leaves us with a pool of genuine language-specific tokens to
    sample from when building test sequences.
    """
    pools = {}
    for model_key, model in models.items():
        vocab_size    = model.vocab_size
        effective_max = (vocab_size - 1000) if max_token is None else max_token
        special_ids   = set(model.tokenizer.all_special_ids)
        unk_id        = model.tokenizer.unk_token_id

        for lang_key, language_data in language_data_dict.items():
            # Collect every token ID that appears in any sentence for this language
            token_ids = set()
            for _, item in language_data.iterrows():
                ids = model.tokenizer.encode(item["text"], add_special_tokens=False)
                token_ids.update(ids)

            # Apply all filters in one list comprehension
            filtered = [
                t for t in token_ids
                if (min_token < t < effective_max)
                and (t not in special_ids)
                and (t != unk_id)
            ]

            key = f'{model_key}_{lang_key}'
            pools[key] = filtered
            print(f'{model_key} -> {lang_key}: {len(filtered)} tokens in pool')

    return pools


In [7]:
def build_repeated_sequence(token_pools: dict,
                            batch_size: int = 1,
                            seq_len:    int = 25,
                            n_repeats:  int = 4) -> dict:
    """
    Build repeated-token sequences for the induction-head test.

    For each language pool:
      1. Sample seq_len random tokens from the pool
      2. Repeat that block n_repeats times end-to-end

    Example (seq_len=3, n_repeats=3):
        sample   : [42, 7, 99]
        repeated : [42, 7, 99, 42, 7, 99, 42, 7, 99]

    An induction head attending to the "previous occurrence" offset of
    (seq_len - 1) = 2 positions back can predict the next token correctly.
    """
    language_sequences = {}

    for key, pool in token_pools.items():
        if not pool:
            print(f'Warning: Token pool for "{key}" is empty -- skipping.')
            language_sequences[key] = torch.empty(batch_size, seq_len * n_repeats, dtype=torch.long)
            continue

        pool_tensor = torch.tensor(pool)

        # Sample random indices into the pool, then retrieve the actual token IDs
        indices  = torch.randint(0, len(pool), (batch_size, seq_len))
        rand_seq = torch.gather(
            pool_tensor.unsqueeze(0).expand(batch_size, -1),
            dim=1,
            index=indices
        )

        # Tile the base block n_repeats times along the sequence dimension
        repeated_seq = einops.repeat(rand_seq, "batch seq -> batch (n_repeats seq)", n_repeats=n_repeats)
        language_sequences[key] = repeated_seq

    return language_sequences


In [8]:
def prepend_bos_prefix(models: dict, repeated_sequences: dict) -> dict:
    """
    Prepend the BOS (beginning-of-sequence) token to every sequence.

    Most transformers expect BOS as the first token. Without it, attention
    patterns in the first layer can be unpredictable.
    """
    updated_sequences = {}

    for key, sequences in repeated_sequences.items():
        # Key format: '{model_key}_{lang_key}' -- split on the last underscore
        model_key, lang_key = key.rsplit("_", maxsplit=1)
        model = models[model_key]

        bos_prefix = torch.full(
            (sequences.shape[0], 1),
            model.tokenizer.bos_token_id,
            dtype=torch.long
        )
        updated_sequences[key] = torch.cat([bos_prefix, sequences], dim=1)

    return updated_sequences


In [9]:
def dataset_pipeline(model_ids: dict, languages: dict,
                     dataset: str, split: str) -> Tuple[dict, dict]:
    """Run the full data-preparation pipeline end-to-end."""
    language_data      = load_hf_dataset(dataset, languages, split)
    models             = load_models(model_ids)
    token_pools        = build_token_pools(models, language_data)
    repeated_sequences = build_repeated_sequence(token_pools)
    full_sequences     = prepend_bos_prefix(models, repeated_sequences)
    return models, full_sequences


### 3.1 Log in to HuggingFace and load data

You need a (free) HuggingFace account to download the model weights. Run the cell below and paste your access token when prompted.


In [10]:
huggingface_hub.login()

Now run the full pipeline. This will download the model and build the test sequences.

In [11]:
models, dataset = dataset_pipeline(MODEL_IDS, LANGUAGES, DATASET, SPLIT)

Loaded: tiny-aya
  Device:          cuda:0
  Layers:          36
  Hidden size:     2048
  Attention heads: 16
  Vocabulary size: 262144
tiny-aya -> eng: 6424 tokens in pool
tiny-aya -> yor: 2945 tokens in pool


---
## 4. Induction Score

### How the score works

For a sequence that repeats a block of `seq_len` tokens, an induction head should attend to the token `seq_len - 1` positions ago (the previous occurrence).

In the attention matrix, this corresponds to the **sub-diagonal at offset `-(seq_len-1)`**. We average the attention weight along that diagonal to get one head's induction score. A score near 1 = reliably 'looks back'; near 0 = does not.

```
Attention matrix for [A B C | A B C | A B C] (seq_len=3, offset=-2):

         A  B  C  A  B  C  A  B  C
     A [ .                        ]
     B [ .  .                     ]
     C [ .  .  .                  ]
     A [ .  .  .  .               ]
     B [ .  .  . [*] .            ]  <- B attends to previous B (offset -3)
     C [ .  .  .  . [*] .         ]  <- C attends to previous C
              ^^^^^^^^^
              sub-diagonal at -(seq_len-1)
```

We compute this score for every layer and every head, then display a heatmap.


In [12]:
def compute_induction_scores_batched(
    current_model,
    sequences,
    threshold: float = 0.4,  # kept for API compatibility, not used in scoring
    batch_size: int = 1
):
    """
    Compute induction scores for every (layer, head) pair.

    Returns:
        attention_heads  -- list of "layer.head" strings for axis labels
        score_accumulator -- FloatTensor (n_layers, n_q_heads)
    """
    n_q_heads  = current_model.config.num_attention_heads
    n_kv_heads = current_model.config.num_key_value_heads
    n_groups   = n_q_heads // n_kv_heads  # GQA: number of query heads per key/value head
    d_head     = current_model.config.hidden_size // n_q_heads

    score_accumulator = torch.zeros(current_model.num_layers, n_q_heads)

    for i in range(0, len(sequences), batch_size):
        batch   = sequences[i:i + batch_size]
        seq_len = batch.shape[1]

        # Forward pass: cache only Q and K projections, offload to CPU immediately
        # to keep GPU memory free. We discard all other activations.
        with torch.no_grad():
            with current_model.trace(batch) as tracer:
                cache = tracer.cache(
                    modules=[
                        current_model.model.layers[l].self_attn.q_proj
                        for l in range(current_model.num_layers)
                    ] + [
                        current_model.model.layers[l].self_attn.k_proj
                        for l in range(current_model.num_layers)
                    ],
                    device=torch.device('cpu'),
                    dtype=torch.float16,
                )

        for layer in range(current_model.num_layers):
            q_key = f'model.model.layers.{layer}.self_attn.q_proj'
            k_key = f'model.model.layers.{layer}.self_attn.k_proj'

            Q = cache[q_key].output.float()   # (batch, seq, n_q_heads * d_head)
            K = cache[k_key].output.float()   # (batch, seq, n_kv_heads * d_head)

            # Reshape into (batch, heads, seq, d_head) for attention computation
            Q = einops.rearrange(Q, 'b s (h d) -> b h s d', h=n_q_heads)
            K = einops.rearrange(K, 'b s (h d) -> b h s d', h=n_kv_heads)

            # Expand K to match query head count (Grouped-Query Attention)
            K = einops.repeat(K, 'b h s d -> b (h g) s d', g=n_groups)

            # Scaled dot-product: (batch, heads, query_pos, key_pos)
            attn_scores = einops.einsum(Q, K, 'b h q d, b h k d -> b h q k') / (d_head ** 0.5)
            del Q, K

            # Causal mask: each token can only attend to previous tokens (not future)
            causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
            attn_scores = attn_scores.masked_fill(~causal_mask, float('-inf'))
            del causal_mask

            # Softmax converts raw scores into probabilities
            attn_pattern = F.softmax(attn_scores, dim=-1)
            del attn_scores

            # One repeat block is (seq_len - 1) // 4 tokens long
            seq_chunk = (seq_len - 1) // 4

            for head in range(n_q_heads):
                pattern = attn_pattern[:, head, :, :]
                # Average attention weight along the induction sub-diagonal
                score = pattern.diagonal(offset=-(seq_chunk - 1), dim1=-2, dim2=-1).mean()
                score_accumulator[layer, head] += score.item()

            del attn_pattern

        del cache
        torch.cuda.empty_cache()
        print(f'Processed {min(i + batch_size, len(sequences))}/{len(sequences)} sequences')

    # Divide by number of sequences to get the mean score
    score_accumulator /= len(sequences)

    attention_heads = [f"{layer}.{head}"
                       for layer in range(current_model.num_layers)
                       for head  in range(n_q_heads)]

    return attention_heads, score_accumulator


---
## 5. Visualising Attention Patterns

`circuitsvis` renders an interactive attention heatmap where:

- **Rows** = query positions (token being processed)
- **Columns** = key positions (tokens being attended to)
- **Brightness** = attention weight

An induction head shows a bright stripe just *below* the main diagonal, offset by exactly `seq_chunk - 1` steps to the left.


In [13]:
def get_attention_pattern_for_layer(
    current_model,
    sequence,
    layer: int,
    heads: list
) -> torch.Tensor:
    """
    Compute the full attention pattern for selected heads in one layer.

    Returns FloatTensor of shape (len(heads), seq_len, seq_len).
    attn[i, q, k] = probability that heads[i] at query position q attends to key position k.
    """
    n_q_heads  = current_model.config.num_attention_heads
    n_kv_heads = current_model.config.num_key_value_heads
    n_groups   = n_q_heads // n_kv_heads
    d_head     = current_model.config.hidden_size // n_q_heads

    # Only cache the two projections we need for this single layer
    with torch.no_grad():
        with current_model.trace(sequence) as tracer:
            cache = tracer.cache(
                modules=[
                    current_model.model.layers[layer].self_attn.q_proj,
                    current_model.model.layers[layer].self_attn.k_proj,
                ],
                device=torch.device('cpu'),
                dtype=torch.float16,
            )

    seq_len = sequence.shape[1]

    Q = cache[f'model.model.layers.{layer}.self_attn.q_proj'].output.float()
    K = cache[f'model.model.layers.{layer}.self_attn.k_proj'].output.float()
    del cache

    Q = einops.rearrange(Q, 'b s (h d) -> b h s d', h=n_q_heads)
    K = einops.rearrange(K, 'b s (h d) -> b h s d', h=n_kv_heads)
    K = einops.repeat(K, 'b h s d -> b (h g) s d', g=n_groups)

    attn_scores = einops.einsum(Q, K, 'b h q d, b h k d -> b h q k') / (d_head ** 0.5)
    del Q, K

    causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    attn_scores = attn_scores.masked_fill(~causal_mask, float('-inf'))
    del causal_mask

    attn_pattern = F.softmax(attn_scores, dim=-1)
    del attn_scores

    # Select only the requested heads from batch index 0
    selected = attn_pattern[0, heads, :, :].clone()
    del attn_pattern
    torch.cuda.empty_cache()

    return selected  # shape: (len(heads), seq_len, seq_len)


In [14]:
def visualise_induction_heads(
    current_model,
    sequence,
    induction_heads: list,
    lang: str = "english"
):
    """
    Display interactive attention visualisations for a list of induction heads.

    Heads are passed as "layer.head" strings (e.g. "3.7").
    We group by layer to minimise forward passes.
    """
    # Group head indices by layer
    heads_by_layer = defaultdict(list)
    for head_str in induction_heads:
        layer, head = head_str.split(".")
        heads_by_layer[int(layer)].append(int(head))

    for layer, heads in sorted(heads_by_layer.items()):
        attn_pattern = get_attention_pattern_for_layer(current_model, sequence, layer, heads)

        # Decode each token ID individually to handle multi-byte unicode correctly
        str_tokens = [
            current_model.tokenizer.decode([token_id])
            for token_id in sequence[0].tolist()
        ]

        print(f'\nLayer {layer} -- induction heads {heads} -- {lang}')
        display(cv.attention.attention_patterns(
            tokens=str_tokens,
            attention=attn_pattern,
        ))

        torch.cuda.empty_cache()


### 5.1 Run induction scoring and plot heatmaps

For each (model, language) pair:
1. Compute induction scores for all heads
2. Plot a heatmap -- warm/red = high induction score


In [15]:
for model_lang_key, sequences in dataset.items():
    model_key, lang_key = model_lang_key.rsplit("_", maxsplit=1)
    current_model = models[model_key]

    gc.collect()
    torch.cuda.empty_cache()

    heads, scores = compute_induction_scores_batched(current_model, sequences, batch_size=1)

    n_layers = scores.shape[0]
    n_heads  = scores.shape[1]

    fig = px.imshow(
        scores.numpy(),
        labels={"x": "Head", "y": "Layer", "color": "Induction score"},
        title=f"Induction score by head -- {model_key} / {lang_key}",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        text_auto=".3f",
        width=900,
        height=max(600, n_layers * 30),
        x=[str(h) for h in range(n_heads)],
        y=[str(l) for l in range(n_layers)],
    )
    fig.update_layout(
        xaxis=dict(tickmode='linear', tick0=0, dtick=1, title='Head'),
        yaxis=dict(tickmode='linear', tick0=0, dtick=1, title='Layer'),
        font=dict(size=10),
    )
    fig.show()

    torch.cuda.empty_cache()


Processed 1/1 sequences


Processed 1/1 sequences


### 5.2 Visualise attention patterns of high-scoring heads

Collect any head with an induction score above 0.4 and display its attention pattern. Look for the bright stripe below the main diagonal.


In [16]:
INDUCTION_THRESHOLD = 0.4  # heads with score above this are visualised

for model_lang_key, sequences in dataset.items():
    model_key, lang_key = model_lang_key.rsplit("_", maxsplit=1)
    current_model = models[model_key]

    gc.collect()
    torch.cuda.empty_cache()

    heads, scores = compute_induction_scores_batched(current_model, sequences, batch_size=1)

    n_layers = scores.shape[0]
    n_heads  = scores.shape[1]

    # Collect heads whose induction score exceeds the threshold
    high_score_heads = [
        f'{layer}.{head}'
        for layer in range(n_layers)
        for head  in range(n_heads)
        if scores[layer, head] > INDUCTION_THRESHOLD
    ]

    print(f'{model_lang_key}: {len(high_score_heads)} heads above threshold {INDUCTION_THRESHOLD}')

    # Pass only the first sequence for the visualiser
    visualise_induction_heads(
        current_model,
        sequences[:1],
        induction_heads=high_score_heads,
        lang=lang_key
    )

    torch.cuda.empty_cache()


Processed 1/1 sequences
tiny-aya_eng: 5 heads above threshold 0.4

Layer 7 -- induction heads [4, 7] -- eng



Layer 19 -- induction heads [10] -- eng



Layer 23 -- induction heads [11] -- eng



Layer 27 -- induction heads [8] -- eng


Processed 1/1 sequences
tiny-aya_yor: 8 heads above threshold 0.4

Layer 3 -- induction heads [8] -- yor



Layer 6 -- induction heads [5] -- yor



Layer 7 -- induction heads [4, 7] -- yor



Layer 19 -- induction heads [10] -- yor



Layer 23 -- induction heads [11] -- yor



Layer 27 -- induction heads [3, 8] -- yor


---
## 6. Ablation Study

### What is ablation?

We *ablate* a head by setting its output to zero during a forward pass. If loss goes up after zeroing a head, that head was doing something useful.

```
ablation_score[layer, head] = loss_with_head_zeroed - baseline_loss
```

- **Positive score** -> ablating this head hurts performance -> it is *useful*
- **Negative score** -> ablating this head helps performance -> it was *harmful* (rare)
- **Near zero**      -> the head does not matter

We measure loss only on the **last repetition** of the sequence, because that is where in-context learning is actually being tested.


In [17]:
def compute_log_probability(logits, tokens):
    """
    Compute the log-probability the model assigns to each correct next token.

    For sequence [t0, t1, ..., tN], the model predicts the next token at each position.
    We compare prediction at position i with the true token at position i+1.

    Returns FloatTensor of shape (batch, seq_len - 1) -- one value per prediction step.
    """
    # Convert raw logits to log-probabilities over the vocabulary
    log_prob = F.log_softmax(logits.float(), dim=-1)

    # Drop the last logit (no ground truth target for the final position)
    last_token_log_prob = log_prob[:, :-1, :]           # (batch, seq_len-1, vocab)

    # Drop the BOS token from targets; predictions start at position 1
    tokens_no_bos = tokens[:, 1:].unsqueeze(-1)         # (batch, seq_len-1, 1)

    # Gather the log-prob of the correct token at each step
    correct_log_prob = torch.gather(last_token_log_prob, dim=-1, index=tokens_no_bos)
    return correct_log_prob.squeeze(-1)                  # (batch, seq_len-1)


In [18]:
def get_baseline_loss(current_model, sequences, seq_chunk: int) -> float:
    """
    Compute the model's average loss on the last repetition of each sequence,
    with no ablation (all heads fully active).

    We only measure loss on the final seq_chunk tokens because those positions
    require the model to use in-context information to predict correctly.
    """
    baseline_total_loss = 0.0

    for i in range(len(sequences)):
        seq = sequences[i:i+1].to(current_model.device)

        with torch.no_grad():
            with current_model.trace(seq) as tracer:
                baseline_logits = current_model.logits.save()

        baseline_log_prob = compute_log_probability(baseline_logits.cpu(), seq.cpu())

        # Negative log-prob = cross-entropy loss; average over the last seq_chunk positions
        loss = -(baseline_log_prob[:, -(seq_chunk - 1):]).mean()
        baseline_total_loss += loss.item()

        del baseline_logits, baseline_log_prob
        torch.cuda.empty_cache()

    return baseline_total_loss / len(sequences)


In [19]:
def get_ablation_loss(current_model, sequences, baseline_loss: float,
                      seq_chunk: int, layers_heads_tuple=None) -> torch.Tensor:
    """
    Compute the ablation score for every (layer, head) pair.

    For each head we:
      1. Zero out that head's slice of the o_proj input in every forward pass
      2. Compute loss on the last seq_chunk tokens
      3. Subtract the baseline loss -> ablation score

    The o_proj input has shape (batch, seq, num_heads * head_dim).
    Each head occupies a contiguous slice of size head_dim.
    """
    num_layers = current_model.num_layers if layers_heads_tuple is None else layers_heads_tuple[0]
    num_heads  = current_model.num_heads  if layers_heads_tuple is None else layers_heads_tuple[1]
    head_dim   = current_model.hidden_size // num_heads

    ablation_scores = torch.zeros(num_layers, num_heads)

    for layer in tqdm(range(num_layers), desc='Ablating layers'):
        for head in range(num_heads):
            total_loss = 0.0

            for i in range(len(sequences)):
                seq = sequences[i:i+1].to(current_model.device)

                with torch.no_grad():
                    with current_model.trace(seq) as tracer:
                        # Access the input tensor going into the output projection
                        z_out = current_model.model.layers[layer].self_attn.o_proj.inputs[0][0]
                        # Zero out this head's slice (all sequence positions, all batches)
                        z_out[:, :, head * head_dim : (head + 1) * head_dim] = 0.
                        ablation_logits = current_model.logits.save()

                log_prob   = compute_log_probability(ablation_logits.cpu(), seq.cpu())
                loss       = -(log_prob[:, -(seq_chunk - 1):]).mean().item()
                total_loss += loss

                del ablation_logits, log_prob
                torch.cuda.empty_cache()

            mean_loss = total_loss / len(sequences)
            # Positive = ablating this head increased loss = the head was doing useful work
            ablation_scores[layer, head] = mean_loss - baseline_loss

    return ablation_scores


### 6.1 Run the ablation study

This is the most compute-intensive step -- one forward pass per (layer, head) per sequence. On a model with 32 layers and 32 heads this means ~1 000 forward passes per language.


In [20]:
model_lang_ablation_scores = {}
model_lang_baseline_loss   = {}

for model_lang_key, sequences in dataset.items():
    model_key, lang_key = model_lang_key.rsplit("_", maxsplit=1)
    current_model = models[model_key]

    # seq_chunk: number of tokens in one repeated block (excluding BOS)
    seq_chunk = (sequences.shape[1] - 1) // 4

    print(f'\nProcessing {model_lang_key}...')
    print(f'  Total sequence length: {sequences.shape[1]}, seq_chunk: {seq_chunk}')

    baseline_loss  = get_baseline_loss(current_model, sequences, seq_chunk)
    print(f'  Baseline loss: {baseline_loss:.4f}')

    ablation_scores = get_ablation_loss(current_model, sequences, baseline_loss, seq_chunk)

    model_lang_ablation_scores[model_lang_key] = ablation_scores
    model_lang_baseline_loss[model_lang_key]   = baseline_loss

    torch.cuda.empty_cache()



Processing tiny-aya_eng...
  Total sequence length: 101, seq_chunk: 25
  Baseline loss: 0.1049


Ablating layers: 100%|██████████| 36/36 [01:16<00:00,  2.13s/it]



Processing tiny-aya_yor...
  Total sequence length: 101, seq_chunk: 25
  Baseline loss: 0.1226


Ablating layers: 100%|██████████| 36/36 [01:16<00:00,  2.13s/it]


### 6.2 Summary and heatmaps

Print a summary then plot ablation scores as heatmaps. Warm/red cells are heads whose removal most hurt the model.


In [21]:
print('\n--- Ablation Summary ---')
for key, baseline in model_lang_baseline_loss.items():
    scores = model_lang_ablation_scores[key]
    print(f'\n{key}')
    print(f'  Baseline loss:               {baseline:.4f}')
    print(f'  Max ablation delta:          {scores.max():.4f}')
    print(f'  Important heads (score>0.05): {(scores > 0.05).nonzero().tolist()}')



--- Ablation Summary ---

tiny-aya_eng
  Baseline loss:               0.1049
  Max ablation delta:          0.1435
  Important heads (score>0.05): [[0, 5], [0, 15], [5, 11], [11, 12], [31, 9]]

tiny-aya_yor
  Baseline loss:               0.1226
  Max ablation delta:          0.1570
  Important heads (score>0.05): [[0, 0], [5, 11], [23, 11], [27, 3], [27, 8], [27, 9], [31, 3]]


In [22]:
for model_lang_key, ablation_scores in model_lang_ablation_scores.items():
    model_key, lang_key = model_lang_key.rsplit("_", maxsplit=1)
    current_model = models[model_key]

    n_layers = current_model.num_layers
    n_heads  = current_model.config.num_attention_heads

    fig = px.imshow(
        ablation_scores.numpy(),
        labels={"x": "Head", "y": "Layer", "color": "Loss increase after ablation"},
        title=f"Ablation scores -- {model_key} / {lang_key}",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        text_auto=".3f",
        width=900,
        height=max(600, n_layers * 30),
        x=[str(h) for h in range(n_heads)],
        y=[str(l) for l in range(n_layers)],
    )
    fig.update_layout(
        xaxis=dict(tickmode='linear', tick0=0, dtick=1, title='Head'),
        yaxis=dict(tickmode='linear', tick0=0, dtick=1, title='Layer'),
        font=dict(size=10),
    )
    fig.show()


---
## 7. Cross-Language Comparison

Now we ask: **do the same heads matter for English and Yoruba?**

We look for:
- **Shared helpful heads** -- important for *both* languages -> likely a language-agnostic mechanism
- **Language-specific heads** -- important for one language only -> specialised circuit
- **Pearson correlation** -- are the two score matrices similar overall?

A high correlation would suggest the model uses the same in-context learning machinery regardless of language.


In [23]:
from scipy.stats import pearsonr

eng_scores = model_lang_ablation_scores["tiny-aya_eng"]
yor_scores = model_lang_ablation_scores["tiny-aya_yor"]

COMPARISON_THRESHOLD = 0.05  # minimum score to consider a head 'helpful'

# Categorise heads by language and direction of effect
eng_helpful = set(map(tuple, (eng_scores >  COMPARISON_THRESHOLD).nonzero().tolist()))
yor_helpful = set(map(tuple, (yor_scores >  COMPARISON_THRESHOLD).nonzero().tolist()))
eng_harmful = set(map(tuple, (eng_scores < -COMPARISON_THRESHOLD).nonzero().tolist()))
yor_harmful = set(map(tuple, (yor_scores < -COMPARISON_THRESHOLD).nonzero().tolist()))

print(f'Helpful heads -- English:  {sorted(eng_helpful)}')
print(f'Helpful heads -- Yoruba:   {sorted(yor_helpful)}')
print(f'Shared helpful:            {sorted(eng_helpful & yor_helpful)}')
print()
print(f'Harmful heads -- English:  {sorted(eng_harmful)}')
print(f'Harmful heads -- Yoruba:   {sorted(yor_harmful)}')
print(f'Shared harmful:            {sorted(eng_harmful & yor_harmful)}')

# Correlation between the two score matrices (flattened to 1D vectors)
r, p = pearsonr(eng_scores.flatten().numpy(), yor_scores.flatten().numpy())
print(f'\nCorrelation (r={r:.3f}, p={p:.4f})')
print('r close to 1 = very similar circuits; r close to 0 = independent circuits')


Helpful heads -- English:  [(0, 5), (0, 15), (5, 11), (11, 12), (31, 9)]
Helpful heads -- Yoruba:   [(0, 0), (5, 11), (23, 11), (27, 3), (27, 8), (27, 9), (31, 3)]
Shared helpful:            [(5, 11)]

Harmful heads -- English:  []
Harmful heads -- Yoruba:   [(3, 8)]
Shared harmful:            []

Correlation (r=0.273, p=0.0000)
r close to 1 = very similar circuits; r close to 0 = independent circuits


In [24]:
# Compare magnitude and distribution of top scores
print(f'Max helpful score -- English: {eng_scores.max():.4f}')
print(f'Max helpful score -- Yoruba:  {yor_scores.max():.4f}')

eng_top = sorted([eng_scores[l, h].item() for l, h in eng_helpful], reverse=True)
yor_top = sorted([yor_scores[l, h].item() for l, h in yor_helpful], reverse=True)

print(f'\nTop helpful scores -- English: {[f"{s:.3f}" for s in eng_top]}')
print(f'Top helpful scores -- Yoruba:  {[f"{s:.3f}" for s in yor_top]}')

print(f'\nSum of helpful scores -- English: {sum(eng_top):.4f}')
print(f'Sum of helpful scores -- Yoruba:  {sum(yor_top):.4f}')


Max helpful score -- English: 0.1435
Max helpful score -- Yoruba:  0.1570

Top helpful scores -- English: ['0.144', '0.098', '0.069', '0.053', '0.053']
Top helpful scores -- Yoruba:  ['0.157', '0.097', '0.086', '0.075', '0.074', '0.059', '0.050']

Sum of helpful scores -- English: 0.4168
Sum of helpful scores -- Yoruba:  0.5978


---
## 8. Cross-Language Ablation

The previous section told us *which* heads matter for each language. Now we perform the causal test:

> *Does ablating English-important heads hurt Yoruba performance, and vice versa?*

| Hypothesis | Prediction |
|---|---|
| **Shared circuit** | Ablating English heads also hurts Yoruba |
| **Separate circuits** | Ablating English heads leaves Yoruba unaffected |

We also test the **shared heads** as a sanity check -- they should hurt both languages.

**Update the head lists below** based on the output of Section 7.


In [25]:
def get_cross_ablation_loss(current_model, sequences, baseline_loss: float,
                            seq_chunk: int, heads_to_ablate: list) -> float:
    """
    Ablate a specific set of (layer, head) pairs simultaneously and measure
    the resulting change in loss.

    Unlike get_ablation_loss(), this zeros out all specified heads in a single
    forward pass, making it much faster for multi-head experiments.

    Args:
        heads_to_ablate: list of (layer, head) tuples, e.g. [(0, 5), (31, 9)]

    Returns:
        Loss increase relative to baseline (positive = these heads were collectively useful)
    """
    head_dim   = current_model.hidden_size // current_model.num_heads
    total_loss = 0.0

    for i in range(len(sequences)):
        seq = sequences[i:i+1].to(current_model.device)

        with torch.no_grad():
            with current_model.trace(seq) as tracer:
                # Zero out every specified head in a single forward pass
                for layer, head in heads_to_ablate:
                    z_out = current_model.model.layers[layer].self_attn.o_proj.inputs[0][0]
                    z_out[:, :, head * head_dim : (head + 1) * head_dim] = 0.
                ablation_logits = current_model.logits.save()

        log_prob   = compute_log_probability(ablation_logits.cpu(), seq.cpu())
        loss       = -(log_prob[:, -(seq_chunk - 1):]).mean().item()
        total_loss += loss

        del ablation_logits, log_prob
        torch.cuda.empty_cache()

    return (total_loss / len(sequences)) - baseline_loss


### 8.1 Define head groups

Fill in these lists based on the Section 7 output. The example values below are placeholders from an earlier run.


In [26]:
# Update these lists based on the Section 7 output
# Format: (layer_index, head_index)
eng_only_heads = [(0, 5), (0, 15), (31, 9)]          # important for English only
yor_only_heads = [(0, 0), (23, 11), (27, 3), (27, 8), (27, 9)]  # important for Yoruba only
shared_heads   = [(5, 11)]                            # important for both languages

# Retrieve sequences and precomputed baselines
eng_sequences = dataset["tiny-aya_eng"]
yor_sequences = dataset["tiny-aya_yor"]

eng_seq_chunk = (eng_sequences.shape[1] - 1) // 4
yor_seq_chunk = (yor_sequences.shape[1] - 1) // 4

eng_baseline  = model_lang_baseline_loss["tiny-aya_eng"]
yor_baseline  = model_lang_baseline_loss["tiny-aya_yor"]

current_model = models["tiny-aya"]


### 8.2 Run the cross-language ablation experiments


In [27]:
print('=== Cross-language ablation ===\n')

# Experiment 1: ablate English-only heads, measure impact on Yoruba
# If Yoruba is hurt -> the heads are shared despite different ablation signatures
# If Yoruba is fine -> the circuits are truly language-separate
yor_loss_when_eng_ablated = get_cross_ablation_loss(
    current_model, yor_sequences, yor_baseline, yor_seq_chunk, eng_only_heads
)
print(f'Ablating English-only heads | impact on Yoruba:  {yor_loss_when_eng_ablated:+.4f}')

# Experiment 2: ablate Yoruba-only heads, measure impact on English
eng_loss_when_yor_ablated = get_cross_ablation_loss(
    current_model, eng_sequences, eng_baseline, eng_seq_chunk, yor_only_heads
)
print(f'Ablating Yoruba-only heads  | impact on English: {eng_loss_when_yor_ablated:+.4f}')

# Experiment 3: ablate shared head, test on both (sanity check -- should hurt both)
eng_loss_shared = get_cross_ablation_loss(
    current_model, eng_sequences, eng_baseline, eng_seq_chunk, shared_heads
)
yor_loss_shared = get_cross_ablation_loss(
    current_model, yor_sequences, yor_baseline, yor_seq_chunk, shared_heads
)
print(f'\nAblating shared heads | impact on English: {eng_loss_shared:+.4f}')
print(f'Ablating shared heads | impact on Yoruba:  {yor_loss_shared:+.4f}')

print('\nInterpretation guide:')
print('  >= +0.05  meaningful loss increase  (head was useful for this language)')
print('  ~  0.00   no impact                 (head irrelevant for this language)')
print('  negative  loss decreased             (head was actively harmful -- rare)')


=== Cross-language ablation ===

Ablating English-only heads | impact on Yoruba:  -0.0189
Ablating Yoruba-only heads  | impact on English: +0.3510

Ablating shared heads | impact on English: +0.0691
Ablating shared heads | impact on Yoruba:  +0.1570

Interpretation guide:
  >= +0.05  meaningful loss increase  (head was useful for this language)
  ~  0.00   no impact                 (head irrelevant for this language)
  negative  loss decreased             (head was actively harmful -- rare)
